# Cross-language SVA circuit dashboard

This dashboard reads only versioned dataset metadata and compact JSON/CSV summaries. Missing formal jobs are shown as pending instead of raising an error.

In [ ]:
from pathlib import Path
import csv
import json
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'configs').is_dir() and (candidate / 'src/mllms').is_dir():
            return candidate
    raise FileNotFoundError('Run from inside the mllms-colab repository')

ROOT = find_repo_root()
DATASET_META = ROOT / 'data/sva/controlled_v2/metadata.json'
SVA_ROOT = ROOT / 'outputs/evaluation/sva_wikipedia_v2'
STATISTICS = {
    'opposite-number': ROOT / 'outputs/interpretability/wikipedia_cross_language_v2_confirm/statistics/prediction_head_statistics.csv',
    'opposite-number-shuffled': ROOT / 'outputs/interpretability/wikipedia_cross_language_v2_confirm/statistics_opposite_number_shuffled/prediction_head_statistics.csv',
}
metadata = json.loads(DATASET_META.read_text())
display(Markdown(
    f'**Dataset:** `{metadata["dataset"]}`  \n'
    f'**Tokenizer SHA-256:** `{metadata["tokenizer_sha256"]}`  \n'
    f'**Dev/Test pairs:** {metadata["splits"]["dev"]["num_pairs"]:,} / {metadata["splits"]["test"]["num_pairs"]:,}'
))

In [ ]:
summary_paths = sorted(SVA_ROOT.glob('*/sva_summary.json'))
if not summary_paths:
    display(Markdown('## SVA trajectory\nFormal v2 SVA jobs are pending.'))
else:
    summaries = [json.loads(path.read_text()) for path in summary_paths]
    summaries.sort(key=lambda row: row['checkpoint_step'])
    steps = [row['checkpoint_step'] for row in summaries]
    fig, axis = plt.subplots(figsize=(8, 4))
    for language in ('original', 'clone'):
        axis.plot(steps, [row['overall'][language]['accuracy'] for row in summaries], marker='o', label=language.title())
    axis.axhline(.5, color='black', linestyle=':')
    axis.set(xlabel='Optimizer step', ylabel='Prompt accuracy', title='Controlled SVA v2 trajectory', ylim=(0, 1))
    axis.legend()
    plt.show()

In [ ]:
if not all(path.is_file() for path in STATISTICS.values()):
    display(Markdown('## Confirmatory L8H3 analysis\nFormal cross-language statistics are pending.'))
else:
    table = ['| Null | Direction | Task | Effect vs null | 95% CI | Holm p | N |', '|---|---|---|---:|---:|---:|---:|']
    for null_name, path in STATISTICS.items():
        rows = list(csv.DictReader(path.open()))
        for row in rows:
            if row['layer'] != '8' or row['head'] != '3':
                continue
            table.append(
                f'| {null_name} | {row["direction"]} | {row["task"]} | {float(row["mean_difference"]):.4f} | '
                f'[{float(row["ci_low"]):.4f}, {float(row["ci_high"]):.4f}] | {row["holm_p"] or "—"} | {row["num_examples"]} |'
            )
    display(Markdown('## Confirmatory L8H3 analysis\n' + '\n'.join(table)))